In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F

#locks down every source of randomness for python, numpy, pytorch
def set_global_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as e:
            print("Note: full determinism unavailable in this torch build:", e)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


set_global_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: full determinism unavailable in this torch build: torch._inductor.config.deterministic does not exist
Device: cpu


In [ ]:
#calculates knowledge distillation loss
def vanilla_kd_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    targets: torch.Tensor,
    temperature: float = 4.0,
    alpha: float = 0.9,
):
    # Hard-label supervision (student vs ground truth)
    hard_loss = F.cross_entropy(student_logits, targets)

    # Soft-label distillation (student mimics teacher's softened distribution)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1) #student soft probs
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    soft_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    ) * (temperature ** 2)

    # Weighted blend of both signals
    total_loss = alpha * soft_loss + (1.0 - alpha) * hard_loss

    return total_loss, hard_loss.detach(), soft_loss.detach()

In [ ]:
# Sanity check 
torch.manual_seed(0)

batch_size, num_classes = 4, 10
fake_student = torch.randn(batch_size, num_classes)
fake_teacher = torch.randn(batch_size, num_classes)
fake_targets = torch.randint(0, num_classes, (batch_size,))

total, hard, soft = vanilla_kd_loss(
    fake_student, fake_teacher, fake_targets,
    temperature=4.0, alpha=0.9,
)

print(f"Total loss: {total.item():.4f}")
print(f"Hard  loss: {hard.item():.4f}")
print(f"Soft  loss: {soft.item():.4f}")

# Sanity property: if student == teacher, soft loss should be ~0
_, _, soft_same = vanilla_kd_loss(
    fake_teacher, fake_teacher, fake_targets,
    temperature=4.0, alpha=0.9,
)
print(f"Soft loss when student==teacher (expect ~0): {soft_same.item():.6f}")

Total loss: 1.0906
Hard  loss: 2.6152
Soft  loss: 0.9212
Soft loss when student==teacher (expect ~0): 0.000000


In [ ]:
import torch.nn as nn
from torchvision.models import resnet34, resnet18


def adapt_resnet_for_cifar(model: nn.Module, num_classes: int = 10) -> nn.Module:
    # Replace ImageNet 7x7 stride-2 stem with a CIFAR-friendly 3x3 stride-1 stem
    model.conv1 = nn.Conv2d(
        in_channels=3, out_channels=64,
        kernel_size=3, stride=1, padding=1, bias=False,
    )
    # Remove the initial maxpool (too aggressive for 32x32 inputs)
    model.maxpool = nn.Identity()
    # Resize the final classifier to our number of classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_teacher(num_classes: int = 10) -> nn.Module:
    model = resnet34(weights=None)          # random init (we train from scratch)
    return adapt_resnet_for_cifar(model, num_classes)


# Quick check: build and inspect output shape
teacher = build_teacher(num_classes=10).to(device)
dummy = torch.randn(2, 3, 32, 32).to(device)
out = teacher(dummy)
print("Teacher output shape:", out.shape)  

Teacher output shape: torch.Size([2, 10])


In [20]:
def build_student(num_classes: int = 10) -> nn.Module:
    model = resnet18(weights=None)          # from scratch, same as teacher
    return adapt_resnet_for_cifar(model, num_classes)


# Quick check: build and inspect output shape + parameter counts
student = build_student(num_classes=10).to(device)
out = student(dummy)
print("Student output shape:", out.shape)   # expect [2, 10]

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

teacher_params = count_params(teacher)
student_params = count_params(student)
print(f"Teacher params: {teacher_params:,}")
print(f"Student params: {student_params:,}")
print(f"Compression ratio (teacher/student): {teacher_params / student_params:.2f}x")

Student output shape: torch.Size([2, 10])
Teacher params: 21,282,122
Student params: 11,173,962
Compression ratio (teacher/student): 1.90x


In [21]:
import torchvision.transforms as T

# Precomputed CIFAR-10 channel statistics (standard values from literature)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Training transform: light augmentation + normalization
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Eval transform: NO augmentation, only normalization
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

print("Transforms ready.")

Transforms ready.


In [ ]:
#ONLY ONCE

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10

DATA_ROOT = "./data"
BATCH_SIZE = 128
VAL_SIZE = 5000          # carved out of the 50k training images
NUM_WORKERS = 2

# Load the training pool twice with different transforms, same underlying images
train_pool_aug = CIFAR10(DATA_ROOT, train=True, download=True, transform=train_transform)
train_pool_eval = CIFAR10(DATA_ROOT, train=True, download=True, transform=eval_transform)

# Official test set (never touched during training/tuning)
test_set = CIFAR10(DATA_ROOT, train=False, download=True, transform=eval_transform)

# Reproducible split of indices 0..49999 into train / val
num_train_total = len(train_pool_aug)                       # 50000
generator = torch.Generator().manual_seed(42)               # fixed split, every run
perm = torch.randperm(num_train_total, generator=generator).tolist()
val_indices = perm[:VAL_SIZE]
train_indices = perm[VAL_SIZE:]

# Train uses augmented pool; val uses eval pool (same images, no augmentation)
train_set = Subset(train_pool_aug, train_indices)
val_set = Subset(train_pool_eval, val_indices)

# DataLoaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, drop_last=False)

print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")

100.0%


Train: 45000  Val: 5000  Test: 10000
